# 11 · Manifest di evidenza e delivery gate

Hash canonici, artefatti, snapshot read-only e autorizzazione separata. Notebook
autocontenuto, offline, standard library.

## Obiettivi, prerequisiti e modalità di lettura

Costruirai una catena di evidenza, rileverai tampering e separerai successo da delivery. Durata: 25–35 minuti. Tutto gira offline.

Ogni blocco di codice è preceduto da una spiegazione e seguito da un **output
atteso**. Quando interviene un modello, l'output atteso descrive proprietà e
invarianti, non una frase letterale. Esegui le celle in ordine e non saltare i
casi negativi: mostrano il confine del meccanismo, non un incidente del corso.

## 1 · Manifest canonico

### Spiegazione del blocco · Costruzione del manifest

JSON canonico e SHA-256 collegano goal, risposta e artefatti. Ordinamento stabile rende il digest riproducibile.

In [ ]:
import hashlib, json, shutil, stat
from pathlib import Path
from tempfile import TemporaryDirectory

def digest_bytes(value: bytes) -> str:
    return hashlib.sha256(value).hexdigest()

def canonical(value: dict) -> bytes:
    return json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=False).encode()

def build_manifest(goal: str, answer: str, workspace: Path, artifacts: list[str]) -> dict:
    manifest = {
        "goal_sha256": digest_bytes(goal.encode()),
        "answer_sha256": digest_bytes(answer.encode()),
        "artifacts": [{"path": name, "sha256": digest_bytes((workspace / name).read_bytes())} for name in artifacts],
        "terminal_status": "completed",
    }
    manifest["manifest_sha256"] = digest_bytes(canonical(manifest))
    return manifest

### Output atteso

Nessun output. Helper per digest e manifest sono definiti.

## 2 · Verifica e rilevamento manomissione

### Spiegazione del blocco · Tamper detection

La prima verifica passa; dopo la modifica del file lo stesso manifest non corrisponde più al workspace.

In [ ]:
def verify(manifest: dict, workspace: Path) -> bool:
    payload = {k: v for k, v in manifest.items() if k != "manifest_sha256"}
    if digest_bytes(canonical(payload)) != manifest["manifest_sha256"]:
        return False
    return all((workspace / item["path"]).is_file() and digest_bytes((workspace / item["path"]).read_bytes()) == item["sha256"] for item in manifest["artifacts"])

with TemporaryDirectory() as temporary:
    root = Path(temporary); workspace = root / "workspace"; workspace.mkdir()
    (workspace / "report.txt").write_text("verificato", encoding="utf-8")
    manifest = build_manifest("crea report", "fatto", workspace, ["report.txt"])
    assert verify(manifest, workspace)
    (workspace / "report.txt").write_text("alterato", encoding="utf-8")
    assert not verify(manifest, workspace)
    print("Tampering rilevato")

### Output atteso

`Tampering rilevato`; entrambi gli assert passano.

## 3 · Checker su snapshot separato

### Spiegazione del blocco · Snapshot indipendente

Il checker legge una copia read-only. Una modifica successiva al workspace non altera retroattivamente ciò che è stato controllato.

In [ ]:
with TemporaryDirectory() as temporary:
    root = Path(temporary); workspace = root / "workspace"; bundle = root / "bundle"
    workspace.mkdir(); bundle.mkdir()
    artifact = workspace / "report.txt"; artifact.write_text("versione approvata")
    manifest = build_manifest("crea report", "fatto", workspace, ["report.txt"])
    shutil.copy2(artifact, bundle / "report.txt")
    (bundle / "report.txt").chmod(stat.S_IRUSR | stat.S_IRGRP | stat.S_IROTH)
    snapshot_ok = digest_bytes((bundle / "report.txt").read_bytes()) == manifest["artifacts"][0]["sha256"]
    artifact.write_text("nuova versione")
    assert snapshot_ok and not verify(manifest, workspace)
    print("Checker separato:", snapshot_ok)

### Output atteso

`Checker separato: True`.

## 4 · Successo tecnico ≠ permesso di deploy

### Spiegazione del blocco · Gate di delivery

Successo tecnico non basta per pubblicare. Il gate richiede congiunzione di integrità, CI, preview, rollback e decisione umana.

In [ ]:
def delivery_ready(*, integrity, checker, branch, clean, ci, preview, rollback, approved):
    return all([integrity, checker, branch not in {"main", "master", ""}, clean, ci, preview, rollback, approved])

assert not delivery_ready(integrity=True, checker=True, branch="feature/x", clean=True, ci=False, preview=False, rollback=True, approved=True)
assert delivery_ready(integrity=True, checker=True, branch="feature/x", clean=True, ci=True, preview=True, rollback=True, approved=True)
print("Gate verificato")

### Output atteso

`Gate verificato`; scenario incompleto fallisce e scenario completo passa.

## Prova tu

Lega approvazione a `manifest_sha256`; una modifica deve invalidarla automaticamente.

## Laboratorio aggiuntivo

Gli esempi seguenti riusano quanto costruito sopra. Il primo amplia il caso normale; il
secondo esercita un confine, un errore o una proprietà che spesso causa bug reali.

## Esempio aggiuntivo: approvazione legata all'hash

### Spiegazione del blocco

Una decisione umana deve valere solo per la versione esatta del dossier controllato.

In [ ]:
def approval_valid(gate: dict, manifest: dict) -> bool:
    return gate.get("decision") == "approved" and gate.get("manifest_sha256") == manifest["manifest_sha256"]

with TemporaryDirectory() as temporary:
    workspace = Path(temporary)
    (workspace / "a.txt").write_text("v1")
    manifest = build_manifest("deploy", "pronto", workspace, ["a.txt"])
    gate = {"decision": "approved", "manifest_sha256": manifest["manifest_sha256"]}
    print("prima:", approval_valid(gate, manifest))
    manifest_modificato = {**manifest, "manifest_sha256": "nuovo-hash"}
    print("dopo modifica:", approval_valid(gate, manifest_modificato))

### Output atteso

`prima: True` e `dopo modifica: False`.

## Esempio aggiuntivo: gate quasi completo

### Spiegazione del blocco

Il caso più pericoloso è assumere che sette condizioni su otto siano sufficienti.

In [ ]:
condizioni = dict(integrity=True, checker=True, branch="feature/x", clean=True, ci=True, preview=True, rollback=True, approved=False)
print("Senza approvazione:", delivery_ready(**condizioni))
condizioni["approved"] = True
print("Con approvazione:", delivery_ready(**condizioni))

### Output atteso

`False` senza approvazione e `True` dopo aver soddisfatto anche l'ultima condizione.

## Riepilogo e troubleshooting

Prima di proseguire, prova a spiegare con parole tue: quale stato è cambiato, quale
componente ha preso la decisione e quale prova rende osservabile l'esito.

Se una cella fallisce:

1. rileggi l'output atteso e individua la prima invariante non rispettata;
2. verifica di aver eseguito tutte le celle precedenti nello stesso kernel;
3. per i notebook live, controlla `.env`, modello disponibile e quota API;
4. riavvia il kernel solo dopo aver conservato eventuali file che vuoi ispezionare;
5. non correggere un caso negativo: l'errore previsto è parte dell'esempio.